# 03 — Maps
Generate and interactively tweak publication-quality distribution maps.

**Run this after** `scripts/run_pipeline.py --dataset both --skip-download --skip-preprocess --skip-fit` (or after full pipeline).

**Kernel:** Python (atmo)

In [ ]:
import sys
sys.path.insert(0, '..')

import yaml
import numpy as np
import matplotlib.pyplot as plt
import matplotlib
matplotlib.rcParams['figure.dpi'] = 130

with open('../config/config.yaml') as f:
    cfg = yaml.safe_load(f)

from src.analysis.distribution import load_stats
from src.visualization.maps import (
    plot_best_fit, plot_delta_aic, plot_ks_map,
    plot_pvalue_map, plot_wetday_freq,
    plot_summary_panel, plot_comparison, plot_all
)

print('Ready.')

## 1. Load stats

In [ ]:
ds_cpc = load_stats(cfg, 'cpc')
print('CPC stats:', ds_cpc)

# Check if IMERG is available
try:
    ds_imerg = load_stats(cfg, 'imerg')
    HAS_IMERG = True
    print('\nIMERG stats:', ds_imerg)
except FileNotFoundError as e:
    HAS_IMERG = False
    print(f'\nIMERG not available yet: {e}')

## 2. CPC — individual maps

In [ ]:
# Best-fit distribution (binary: log-normal vs gamma)
path = plot_best_fit(ds_cpc, cfg, 'cpc')
print(f'Saved: {path}')

In [ ]:
# ΔAIC map — main diagnostic
path = plot_delta_aic(ds_cpc, cfg, 'cpc')
print(f'Saved: {path}')

In [ ]:
# KS statistics
path_ln = plot_ks_map(ds_cpc, cfg, 'cpc', 'lognorm')
path_gm = plot_ks_map(ds_cpc, cfg, 'cpc', 'gamma')
print(f'Saved: {path_ln}')
print(f'Saved: {path_gm}')

In [ ]:
# p-value maps
path_pln = plot_pvalue_map(ds_cpc, cfg, 'cpc', 'lognorm')
path_pgm = plot_pvalue_map(ds_cpc, cfg, 'cpc', 'gamma')
print(f'Saved: {path_pln}')
print(f'Saved: {path_pgm}')

In [ ]:
# Wet-day frequency
path = plot_wetday_freq(ds_cpc, cfg, 'cpc')
print(f'Saved: {path}')

## 3. CPC — 6-panel summary figure

In [ ]:
path = plot_summary_panel(ds_cpc, cfg, 'cpc')
print(f'Saved: {path}')

## 4. IMERG maps (if available)

In [ ]:
if HAS_IMERG:
    paths = plot_all(ds_imerg, cfg, 'imerg')
    print(f'Generated {len(paths)} IMERG maps')
else:
    print('IMERG data not available — skipping')

## 5. CPC vs IMERG comparison

In [ ]:
if HAS_IMERG:
    path = plot_comparison(ds_cpc, ds_imerg, cfg)
    print(f'Comparison map: {path}')
else:
    print('Need both CPC and IMERG to produce comparison map')

## 6. Quick statistics summary

In [ ]:
def summarize(ds, label):
    delta = ds['delta_aic'].values
    valid = delta[np.isfinite(delta)]
    pct_ln = 100 * np.mean(valid < 0)
    pct_gm = 100 * np.mean(valid > 0)

    ks_ln = ds['ks_lognorm'].values
    ks_gm = ds['ks_gamma'].values

    print(f"\n{'='*50}")
    print(f"  {label}")
    print(f"{'='*50}")
    print(f"  Grid points with fit : {np.sum(np.isfinite(valid)):,}")
    print(f"  Log-normal better    : {pct_ln:.1f}%")
    print(f"  Gamma better         : {pct_gm:.1f}%")
    print(f"  Mean ΔAIC            : {np.nanmean(valid):.2f}")
    print(f"  Median ΔAIC          : {np.nanmedian(valid):.2f}")
    print(f"  Mean KS (log-normal) : {np.nanmean(ks_ln):.4f}")
    print(f"  Mean KS (gamma)      : {np.nanmean(ks_gm):.4f}")
    pv_ln = ds['pval_lognorm'].values
    pv_gm = ds['pval_gamma'].values
    print(f"  % p>0.05 log-normal  : {100*np.nanmean(pv_ln > 0.05):.1f}% (cannot reject)")
    print(f"  % p>0.05 gamma       : {100*np.nanmean(pv_gm > 0.05):.1f}% (cannot reject)")

summarize(ds_cpc, 'CPC Gauge-Based')
if HAS_IMERG:
    summarize(ds_imerg, 'IMERG v07')